# 輕度認知障礙與失智症EEG特徵提取

In [27]:
from collections import Counter
from sklearn.exceptions import ConvergenceWarning
from scipy.stats import f_oneway, ttest_ind, spearmanr, pearsonr
import mne, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# 設置紀錄為 'WARNING' 或更高，以隱藏不必要的輸出
mne.set_log_level('WARNING')

檔案路徑

In [28]:
path_hc = 'C:/Users/kitt9/OneDrive - National ChengChi University/eeg-rag-llm-diag/dataset_random/sample_test/sample/HC/'
path_mci = 'C:/Users/kitt9/OneDrive - National ChengChi University/eeg-rag-llm-diag/dataset_random/sample_test/sample/MCI/'
path_ad = 'C:/Users/kitt9/OneDrive - National ChengChi University/eeg-rag-llm-diag/dataset_random/sample_test/sample/AD/'

## 絕對功率

In [29]:
def EEG_calculate_PSD(path, event_name='Eyes Open', delay_time=0, cut_time=8):
    filesname = os.listdir(path)
    total_mean_psd_results = []
    filenames = []

    for filename in filesname:
        file_path = os.path.join(path, filename)

        try:
            raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
        except Exception as e:
            print(f"Error processing file {filename}: {e}")
            continue

        # 提取事件
        events, event_id = mne.events_from_annotations(raw, verbose=False)

        # 選擇事件 (ex. 'Eyes Open')
        try:
            s_events = events[events[:, 2] == event_id[event_name]]
        except Exception as e:
            print(f"No Event Error File {filename}: {e}")
            continue

        # 移除Photic通道
        if 'Photic' in raw.info['ch_names']:
            raw.drop_channels(['Photic'])

        # 定義處理參數
        fs = int(raw.info['sfreq'])  # 取樣率，確保為整數
        filter_low, filter_high = 0.5, 50  # Bandpass Filter範圍

        # 提取ch_names，與建立ch_types
        ch_names = raw.info['ch_names']
        ch_types = ['eeg'] * len(ch_names)  

        # 初始化存儲每個通道的PSD結果的列表
        psd_results = {ch_name: [] for ch_name in ch_names}

        # 遍歷每個 'Eyes Open' 事件
        for s_event in s_events:
            event_start = s_event[0]
            delay = delay_time  # delay_time 預設為 0
            time = cut_time # cut_time 預設為 8
            start_sample = event_start + int(fs * delay)  # 事件後延遲 delay 秒的開始樣本，若沒有要延遲為0
            end_sample = start_sample + int(fs * time)  # time 秒長度的樣本

            if end_sample > len(raw): continue  # 避免超出數據範圍

            # 提取事件後的 time 秒數據
            event_data, _ = raw[:, start_sample:end_sample]
                
            # 濾波
            event_data_filtered = mne.filter.filter_data(event_data, sfreq=fs, l_freq=filter_low, h_freq=filter_high, verbose=False)
            
            # 創建 mne.io.RawArray
            info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types=ch_types)
            epoch_raw = mne.io.RawArray(event_data_filtered, info, verbose=False)
            epoch_data_clean = epoch_raw.get_data()

            # 計算每個通道的PSD
            for ch_idx, ch_name in enumerate(ch_names):
                psd, freqs = mne.time_frequency.psd_array_welch(epoch_data_clean[ch_idx], 
                                                                sfreq=fs, fmin=filter_low, fmax=filter_high, 
                                                                n_fft=int(fs*2), verbose=False)
                psd = psd * (1e6 ** 2) # PSD單位原先為 µV²/Hz 轉換為 V²/Hz
                psd_results[ch_name].append(psd)

        # 計算每個通道所有 epoch 的 PSD 均值和標準差
        mean_psd_results = {ch_name: np.mean(psds, axis=0) for ch_name, psds in psd_results.items()}
        total_mean_psd_results.append(mean_psd_results)
        filenames.append(filename)  # 儲存檔名

    return total_mean_psd_results, filenames, freqs

In [30]:
HC_psd, HC_filenames, freqs = EEG_calculate_PSD(path=path_hc, event_name='Eyes Closed', delay_time=0, cut_time=8)
MCI_psd, MCI_filenames, freqs = EEG_calculate_PSD(path=path_mci, event_name='Eyes Closed', delay_time=0, cut_time=8)
AD_psd, AD_filenames, freqs = EEG_calculate_PSD(path=path_ad, event_name='Eyes Closed', delay_time=0, cut_time=8)

No Event Error File C6444917.edf: 'Eyes Closed'
No Event Error File R3459841.edf: 'Eyes Closed'


In [31]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from scipy.interpolate import make_interp_spline

# -------------------------
# 1) 先定義：把每個檔（同一群組）內「每通道的 epoch 平均 PSD（線性）」轉 dB，再做群組平均/標準差（dB 空間）
# -------------------------
def calculate_mean_std_psd(psd_dict):
    # 僅保留通道數完整的結果（避免 None 或少通道）
    valid_mean_psd_results = [result for result in psd_dict if len(result) == 18]
    ch_names = list(valid_mean_psd_results[0].keys())
    overall_mean_psd = {ch_name: [] for ch_name in ch_names}

    for file_mean_psd in valid_mean_psd_results:
        for ch_name in ch_names:
            # 你的 file_mean_psd[ch_name] 已是「該檔案/通道的 epoch 平均 PSD（線性）」
            # 這裡轉 dB 後再做群組平均（dB 平均）
            overall_mean_psd[ch_name].append(10 * np.log10(file_mean_psd[ch_name]))

    final_mean_psd = {ch_name: np.mean(overall_mean_psd[ch_name], axis=0) for ch_name in ch_names}
    final_std_psd  = {ch_name: np.std (overall_mean_psd[ch_name], axis=0) for ch_name in ch_names}
    return final_mean_psd, final_std_psd

# -------------------------
# 2) 計算三組的群組平均 PSD（dB）與標準差（dB）
# -------------------------
HC_mean_psd, HC_std_psd = calculate_mean_std_psd(HC_psd)
MCI_mean_psd, MCI_std_psd = calculate_mean_std_psd(MCI_psd)
AD_mean_psd,  AD_std_psd  = calculate_mean_std_psd(AD_psd)

# 取所有通道名
ch_names = list(HC_mean_psd.keys())

# -------------------------
# 3) 頻帶選擇（例：0.5–12 Hz）
# -------------------------
desired_freq_band = (0.5, 12)
desired_freq_band_idx = np.where((freqs >= desired_freq_band[0]) & (freqs <= desired_freq_band[1]))[0]

# -------------------------
# 4) 先「一次」算好每個頻率的 t-test（用 dB 值）
# -------------------------
def calculate_ttest_p_values(HC_psd, MCI_psd, AD_psd, ch_names, freq_indices):
    p_values = {ch_name: {'HC_MCI': [], 'HC_AD': [], 'MCI_AD': []} for ch_name in ch_names}
    t_values = {ch_name: {'HC_MCI': [], 'HC_AD': [], 'MCI_AD': []} for ch_name in ch_names}

    for ch_name in ch_names:
        for freq_idx in freq_indices:
            # 從各檔案取出該通道該頻率的 PSD（線性），轉 dB 做統計
            hc_values  = [10 * np.log10(file_psd[ch_name][freq_idx]) for file_psd in HC_psd]
            mci_values = [10 * np.log10(file_psd[ch_name][freq_idx]) for file_psd in MCI_psd]
            ad_values  = [10 * np.log10(file_psd[ch_name][freq_idx]) for file_psd in AD_psd]

            # Welch's t-test（變異數不等）
            t_stat_hc_mci, p_value_hc_mci = ttest_ind(hc_values,  mci_values, equal_var=False)
            t_stat_hc_ad,  p_value_hc_ad  = ttest_ind(hc_values,  ad_values,  equal_var=False)
            t_stat_mci_ad, p_value_mci_ad = ttest_ind(mci_values, ad_values,  equal_var=False)

            p_values[ch_name]['HC_MCI'].append(p_value_hc_mci)
            p_values[ch_name]['HC_AD'].append(p_value_hc_ad)
            p_values[ch_name]['MCI_AD'].append(p_value_mci_ad)

            t_values[ch_name]['HC_MCI'].append(t_stat_hc_mci)
            t_values[ch_name]['HC_AD'].append(t_stat_hc_ad)
            t_values[ch_name]['MCI_AD'].append(t_stat_mci_ad)

    return p_values, t_values

p_values, _ = calculate_ttest_p_values(HC_psd, MCI_psd, AD_psd, ch_names, desired_freq_band_idx)

## Alpha頻帶平均功率

In [32]:
def calculate_band_power(psd, freqs, band):
    band_freqs = [i for i, f in enumerate(freqs) if band[0] <= f < band[1]]
    band_power = {ch: np.mean([psd[ch][i] for i in band_freqs]) for ch in psd}
    return band_power

def calculate_mean_sd(psd_list, freqs, bands):
    all_band_powers = {band: {ch: [] for ch in psd_list[0].keys()} for band in bands}
    
    for psd in psd_list:
        psd_db = {ch: 10 * np.log10(np.array(psd[ch])) for ch in psd}
        for band_name, band_range in bands.items():
            band_power = calculate_band_power(psd_db, freqs, band_range)
            for ch in band_power:
                all_band_powers[band_name][ch].append(band_power[ch])
    
    mean_sd = {band: {ch: (np.mean(all_band_powers[band][ch]), np.std(all_band_powers[band][ch])) for ch in all_band_powers[band]} for band in bands}
    
    return all_band_powers, mean_sd

bands = {
    'Delta': (0.5, 4),
    'Theta': (4, 8),
    'Alpha': (8, 12)
}

# Calculate mean and standard deviation for each group
HC_band_powers, HC_mean_sd = calculate_mean_sd(HC_psd, freqs, bands)
MCI_band_powers, MCI_mean_sd = calculate_mean_sd(MCI_psd, freqs, bands)
AD_band_powers, AD_mean_sd = calculate_mean_sd(AD_psd, freqs, bands)

## Alpha峰值頻率

In [33]:
from scipy.ndimage import uniform_filter1d

def calculate_cgf(psd_values, freqs, alpha_range=(4, 12)):
    psd_values = uniform_filter1d(psd_values, size=5)
    alpha_idx = np.where((freqs >= alpha_range[0]) & (freqs <= alpha_range[1]))[0]
    alpha_psd = psd_values[alpha_idx]
    cgf = np.sum(freqs[alpha_idx] * alpha_psd) / np.sum(alpha_psd)
    return cgf

def calculate_paf(psd_values, freqs, alpha_range=(4, 12)):
    psd_values = uniform_filter1d(psd_values, size=5)
    alpha_idx = np.where((freqs >= alpha_range[0]) & (freqs <= alpha_range[1]))[0]
    alpha_psd = psd_values[alpha_idx]
    peak_idx = np.argmax(alpha_psd)  # 找到最大值的位置
    paf = freqs[alpha_idx][peak_idx]  # 對應的頻率即為PAF
    return paf

def calculate_af_amplitude(psd_values, freqs, af):
    psd_values = np.array(psd_values)
    af_amplitude = np.interp(af, freqs, psd_values) 
    return af_amplitude


## 重心頻率

In [34]:
import numpy as np
from scipy.ndimage import uniform_filter1d

def calculate_center_of_gravity(psd_values, freqs, freq_range=(4, 12), smooth_size=5):
    # 1. 數據平滑處理 (減少雜訊干擾)
    smoothed_psd = uniform_filter1d(psd_values, size=smooth_size)
    
    # 2. 篩選出目標頻段的索引
    mask = (freqs >= freq_range[0]) & (freqs <= freq_range[1])
    target_freqs = freqs[mask]
    target_psd = smoothed_psd[mask]
    
    # 3. 檢查範圍內是否有數據，避免除以零
    if np.sum(target_psd) == 0:
        return np.nan
    
    # 4. 計算重心頻率 (加權平均)
    cgf = np.sum(target_freqs * target_psd) / np.sum(target_psd)
    
    return cgf

# 如果你需要對應的振幅 (Amplitude at CGF)
def get_amplitude_at_freq(psd_values, freqs, target_f):
    return np.interp(target_f, freqs, psd_values)

## 複雜度HDF

In [35]:
import numpy as np

def calculate_hfd(data, k_max=10):
    """
    計算單一通道時域信號的 Higuchi Fractal Dimension (HFD)
    """
    N = len(data)
    L_k = np.zeros(k_max)
    
    for k in range(1, k_max + 1):
        Lk_m = []
        for m in range(k):
            # 建立子序列索引
            indices = np.arange(m, N, k)
            n_max = len(indices)
            
            if n_max > 1:
                # 計算相鄰點的絕對差值和
                # 公式: L_m(k) = [sum |X(i)-X(j)| * (N-1) / (n_max-1 * k)] / k
                diffs = np.abs(np.diff(data[indices]))
                l_m_k = (np.sum(diffs) * (N - 1) / ((n_max - 1) * k)) / k
                Lk_m.append(l_m_k)
        
        # 避免 Lk_m 為空導致 mean 報錯
        L_k[k-1] = np.mean(Lk_m) if Lk_m else 0

    # 移除可能存在的 0 或負值以進行 log 運算
    valid = L_k > 0
    if not np.any(valid): return np.nan
    
    x = np.log(1.0 / np.arange(1, k_max + 1)[valid])
    y = np.log(L_k[valid])
    
    # 線性擬合斜率即為 HFD
    hfd, _ = np.polyfit(x, y, 1)
    return hfd

def batch_calculate_hfd(path, event_name='Eyes Closed', delay_time=0, cut_time=8, k_max=10):
    """
    從資料夾讀取 EDF 並計算每個通道的 HFD 平均值
    """
    filesname = os.listdir(path)
    total_hfd_results = []
    filenames = []

    for filename in filesname:
        file_path = os.path.join(path, filename)
        try:
            raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
            events, event_id = mne.events_from_annotations(raw, verbose=False)
            s_events = events[events[:, 2] == event_id[event_name]]
        except Exception:
            continue

        if 'Photic' in raw.info['ch_names']:
            raw.drop_channels(['Photic'])

        fs = int(raw.info['sfreq'])
        ch_names = raw.info['ch_names']
        
        # 儲存該受試者各通道多個 epoch 的 HFD
        subject_hfd_temp = {ch_name: [] for ch_name in ch_names}

        for s_event in s_events:
            start_sample = s_event[0] + int(fs * delay_time)
            end_sample = start_sample + int(fs * cut_time)

            if end_sample > len(raw): continue

            # 提取數據並濾波
            event_data, _ = raw[:, start_sample:end_sample]
            event_data_filtered = mne.filter.filter_data(event_data, sfreq=fs, l_freq=0.5, h_freq=50, verbose=False)

            for ch_idx, ch_name in enumerate(ch_names):
                hfd_val = calculate_hfd(event_data_filtered[ch_idx], k_max=k_max)
                subject_hfd_temp[ch_name].append(hfd_val)

        # 計算該受試者每個通道在所有 epoch 上的平均 HFD
        subject_mean_hfd = {ch_name: np.mean(vals) for ch_name, vals in subject_hfd_temp.items() if vals}
        
        if subject_mean_hfd:
            total_hfd_results.append(subject_mean_hfd)
            filenames.append(filename)

    return total_hfd_results, filenames

In [36]:
import pandas as pd
from scipy.stats import f_oneway

# --- 1. 計算 HFD (請根據你的實際路徑調整) ---
HC_hfd, HC_hfd_files = batch_calculate_hfd(path_hc)
MCI_hfd, MCI_hfd_files = batch_calculate_hfd(path_mci)
AD_hfd, AD_hfd_files = batch_calculate_hfd(path_ad)

# --- 2. 整理並列印所有 HFD 數值 ---
def convert_to_df(hfd_results, filenames, group_name):
    df = pd.DataFrame(hfd_results)
    df.insert(0, 'Filename', filenames)
    df.insert(1, 'Group', group_name)
    return df

df_hc = convert_to_df(HC_hfd, HC_hfd_files, 'HC')
df_mci = convert_to_df(MCI_hfd, MCI_hfd_files, 'MCI')
df_ad = convert_to_df(AD_hfd, AD_hfd_files, 'AD')

# 合併三組數據
all_hfd_df = pd.concat([df_hc, df_mci, df_ad], ignore_index=True)

print("\n" + "="*30)
print("  所有受試者各通道 HFD 數值表")
print("="*30)
# 使用 pd.set_option 確保可以看到所有欄位
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(all_hfd_df.to_string(index=False))

# --- 3. 進行所有通道的 ANOVA 檢定並列印 ---
print("\n" + "="*30)
print("  所有通道 ANOVA 統計檢定結果")
print("="*30)

# 獲取所有腦波通道名稱
channels = [col for col in all_hfd_df.columns if col not in ['Filename', 'Group']]
anova_summary = []

for ch in channels:
    # 提取三組數據並排除 NaN (確保統計正確)
    group_hc = all_hfd_df[all_hfd_df['Group'] == 'HC'][ch].dropna()
    group_mci = all_hfd_df[all_hfd_df['Group'] == 'MCI'][ch].dropna()
    group_ad = all_hfd_df[all_hfd_df['Group'] == 'AD'][ch].dropna()
    
    # 執行 ANOVA
    if len(group_hc) > 1 and len(group_mci) > 1 and len(group_ad) > 1:
        f_stat, p_val = f_oneway(group_hc, group_mci, group_ad)
        
        anova_summary.append({
            'Channel': ch,
            'F-Statistic': round(f_stat, 4),
            'p-value': round(p_val, 4),
            'Significant': 'YES (p<0.05)' if p_val < 0.05 else 'no'
        })

# 轉換成 DataFrame 並根據 p-value 排序（最顯著的排在前面）
anova_df = pd.DataFrame(anova_summary).sort_values(by='p-value')

print(anova_df.to_string(index=False))



  所有受試者各通道 HFD 數值表
    Filename Group  EEG Fp1-F3  EEG F3-C3  EEG C3-P3  EEG P3-O1  EEG Fp2-F4  EEG F4-C4  EEG C4-P4  EEG P4-O2  EEG Fp1-F7  EEG F7-T3  EEG T3-T5  EEG T5-O1  EEG Fp2-F8  EEG F8-T4  EEG T4-T6  EEG T6-O2  EEG Fz-Cz  EEG Cz-Pz
12340089.edf    HC    1.672780   1.633870   1.598829   1.519882    1.626872   1.595962   1.558575   1.498018    1.631071   1.681718   1.622416   1.567293    1.619538   1.595545   1.514090   1.536843   1.543858   1.534807
B1949126.edf    HC    1.675371   1.783093   1.721112   1.682271    1.663648   1.770654   1.655032   1.617882    1.631825   1.678044   1.639453   1.654128    1.602005   1.707649   1.672783   1.639840   1.705005   1.746511
B2202077.edf    HC    1.804841   1.872312   1.857034   1.777768    1.803433   1.870219   1.848343   1.770328    1.753381   1.790690   1.771158   1.808471    1.735571   1.821915   1.764476   1.822662   1.909481   1.881913
B2257947.edf    HC    1.696847   1.901251   1.896284   1.879977    1.748886   1.899760   1.89899

In [37]:
# --- 4. 計算各組各通道的平均數與變異數 ---
print("\n" + "="*30)
print("  各組各通道 HFD 統計摘要 (平均數 & 變異數)")
print("="*30)

# 1. 執行分組計算
# 我們對 'Group' 分組，並對所有腦波通道 (channels) 套用 mean 和 var 函數
group_stats = all_hfd_df.groupby('Group')[channels].agg(['mean', 'var'])

# 2. 為了方便閱讀，我們可以將結果轉置 (Transpose) 
# 這樣列 (Rows) 會是通道，欄 (Columns) 會是各組的統計數值
group_stats_transposed = group_stats.T

# 3. 列印結果
print(group_stats_transposed)



  各組各通道 HFD 統計摘要 (平均數 & 變異數)
Group                  AD        HC       MCI
EEG Fp1-F3 mean  1.486578  1.518492  1.461732
           var   0.051105  0.044552  0.037858
EEG F3-C3  mean  1.467232  1.499845  1.443665
           var   0.061483  0.069609  0.054325
EEG C3-P3  mean  1.425884  1.433731  1.379174
           var   0.053660  0.078964  0.065260
EEG P3-O1  mean  1.394243  1.379488  1.336829
           var   0.046582  0.079006  0.064380
EEG Fp2-F4 mean  1.495896  1.495309  1.459678
           var   0.045143  0.041509  0.040363
EEG F4-C4  mean  1.478512  1.480551  1.443450
           var   0.050003  0.068049  0.059587
EEG C4-P4  mean  1.418423  1.417366  1.376936
           var   0.037928  0.080550  0.063008
EEG P4-O2  mean  1.378741  1.369892  1.338510
           var   0.042811  0.075623  0.061052
EEG Fp1-F7 mean  1.483946  1.489711  1.454416
           var   0.045811  0.037527  0.033363
EEG F7-T3  mean  1.555305  1.511658  1.503188
           var   0.055354  0.041759  0.051811
EEG 

## 匯入資料庫

In [38]:
import pymysql
import numpy as np
import pandas as pd

# ==========================================================
# 0. 重新整理資料表變數
# ==========================================================
upload_df = all_hfd_df.copy()
upload_df['patient_id'] = upload_df['Filename'].str.replace('.edf', '', case=False)

patients_table = upload_df[['patient_id', 'Group']].drop_duplicates().rename(columns={'Group': 'group_id'})
files_table = upload_df[['patient_id', 'Filename']].rename(columns={'Filename': 'filename'})

# 1. 建立原生 MySQL 連線
conn = pymysql.connect(
    host='localhost',
    user='root',
    password='',
    database='eeg_db_random', 
    port=3306,
    autocommit=True 
)

In [39]:
try:
    print("正在使用 PyMySQL 原生語法匯入資料...")
    with conn.cursor() as cursor:
        
        # 1️⃣ 寫入 patients 表
        for _, row in patients_table.iterrows():
            sql = "INSERT IGNORE INTO patients (patient_id, group_id) VALUES (%s, %s)"
            cursor.execute(sql, (row['patient_id'], row['group_id']))
        print("1. patients 資料表匯入成功！")
        
        # 2️⃣ 寫入 origin_files 表
        for _, row in files_table.iterrows():
            sql = "INSERT IGNORE INTO origin_files (patient_id, filename) VALUES (%s, %s)"
            cursor.execute(sql, (row['patient_id'], row['filename']))
        print("2. origin_files 資料表匯入成功！")
        
        # 3️⃣ 從資料庫抓取自動生成的 file_id 串回 upload_df
        cursor.execute("SELECT file_id, filename FROM origin_files")
        file_mapping = pd.DataFrame(cursor.fetchall(), columns=['file_id', 'Filename'])
        hfd_table_with_id = pd.merge(upload_df, file_mapping, on='Filename', how='left')
        
        # 4️⃣ 【核心修正】：建立「資料庫欄位(底線)」對應「Python欄位(橫線)」的精準對照表
        # 左邊為你 MySQL Workbench 中的欄位名稱，右邊為 Python 輸出的欄位名稱
        mapping_dict = {
            'patient_id': 'patient_id',
            'file_id': 'file_id',
            'Fp1_F3': 'EEG Fp1-F3',
            'F3_C3': 'EEG F3-C3',
            'C3_P3': 'EEG C3-P3',
            'P3_O1': 'EEG P3-O1',
            'Fp2_F4': 'EEG Fp2-F4',
            'F4_C4': 'EEG F4-C4',
            'C4_P4': 'EEG C4-P4',
            'P4_O2': 'EEG P4-O2',
            'Fp1_F7': 'EEG Fp1-F7',
            'F7_T3': 'EEG F7-T3',
            'T3_T5': 'EEG T3-T5',
            'T5_O1': 'EEG T5-O1',
            'Fp2_F8': 'EEG Fp2-F8',
            'F8_T4': 'EEG F8-T4',
            'T4_T6': 'EEG T4-T6',
            'T6_O2': 'EEG T6-O2',
            'Fz_Cz': 'EEG Fz-Cz',
            'Cz_Pz': 'EEG Cz-Pz'
        }
        
        # 5️⃣ 依序讀取每一列進行資料清洗與填入
        success_count = 0
        for _, row in hfd_table_with_id.iterrows():
            insert_data = {}
            
            for db_col, py_col in mapping_dict.items():
                if py_col in row.index:
                    val = row[py_col]
                    # 處理 NaN 值
                    if pd.isna(val) or val is None:
                        insert_data[db_col] = None
                    elif db_col == 'file_id':
                        insert_data[db_col] = int(val)
                    elif db_col == 'patient_id':
                        insert_data[db_col] = str(val)
                    else:
                        insert_data[db_col] = float(val) # 特徵值轉為高精度 float
                else:
                    insert_data[db_col] = None
            
            # 動態動構 SQL 欄位與值 (強制寫入大寫 HFD 表)
            cols = ", ".join([f"`{c}`" for c in insert_data.keys()])
            placeholders = ", ".join(["%s"] * len(insert_data))
            sql = f"INSERT INTO HFD ({cols}) VALUES ({placeholders})"
            
            cursor.execute(sql, tuple(insert_data.values()))
            success_count += 1
            
        print(f"3. HFD 資料表成功寫入 {success_count} 筆完整的特徵數值！")

except Exception as e:
    print(f"匯入失敗，錯誤訊息: {e}")
finally:
    conn.close()

正在使用 PyMySQL 原生語法匯入資料...
1. patients 資料表匯入成功！
2. origin_files 資料表匯入成功！
3. HFD 資料表成功寫入 168 筆完整的特徵數值！


In [ ]:
import pymysql
import numpy as np
import pandas as pd

# ==========================================================
# 1. 抓取前面已算好的功率，並融合個別患者的 PAF 與 CAF (CGF)
# ==========================================================
def collect_individual_alpha_features(group_band_powers, group_psd_list, filenames, freqs):
    alpha_features = []
    alpha_range = (8, 12)
    
    # 這裡的 group_band_powers['Alpha'][ch] 包含了該組所有受試者的 Alpha 功率
    # 我們依受試者 (idx) 拆分，並計算全腦通道的平均值
    for idx, filename in enumerate(filenames):
        patient_id = filename.replace('.edf', '').replace('.EDF', '')
        
        # 1️⃣ 抓取你前面 calculate_mean_sd 算好的各通道 Alpha 功率 (dB)，並取全腦平均
        # 這是利用你前面已經跑好的數據，不再重複執行
        all_ch_powers = []
        for ch in group_band_powers['Alpha'].keys():
            all_ch_powers.append(group_band_powers['Alpha'][ch][idx])
        avg_power = np.mean(all_ch_powers)
        
        # 2️⃣ 同步計算該位患者的 PAF 與 CAF 全腦平均值
        individual_psd = group_psd_list[idx]
        all_ch_paf = []
        all_ch_caf = []
        for ch, psd_val in individual_psd.items():
            all_ch_paf.append(calculate_paf(psd_val, freqs, alpha_range=alpha_range))
            all_ch_caf.append(calculate_cgf(psd_val, freqs, alpha_range=alpha_range))
            
        alpha_features.append({
            'Filename': filename,
            'patient_id': patient_id,
            'average_power': avg_power,
            'PAF': np.mean(all_ch_paf),
            'CAF': np.mean(all_ch_caf)
        })
        
    return pd.DataFrame(alpha_features)

# 直接抓取你前面儲存格跑完的變數進行組裝 (HC_band_powers, HC_psd 等)
df_alpha_hc = collect_individual_alpha_features(HC_band_powers, HC_psd, HC_filenames, freqs)
df_alpha_mci = collect_individual_alpha_features(MCI_band_powers, MCI_psd, MCI_filenames, freqs)
df_alpha_ad = collect_individual_alpha_features(AD_band_powers, AD_psd, AD_filenames, freqs)

# 合併成最終要匯入的表
all_alpha_df = pd.concat([df_alpha_hc, df_alpha_mci, df_alpha_ad], ignore_index=True)

print("--- 已經成功將你前面算好的數據，整理為個別患者的 Alpha 欄位 ---")
print(all_alpha_df.head(10).to_string(index=False))
print("-----------------------------------------------------------\n")

try:
    print("正在將 Alpha 特徵匯入 MySQL 資料庫...")
    with conn.cursor() as cursor:
        
        # 重新抓取對照用的 file_id
        cursor.execute("SELECT file_id, filename FROM origin_files")
        file_mapping = pd.DataFrame(cursor.fetchall(), columns=['file_id', 'Filename'])
        
        # 合併 file_id
        final_alpha_table = pd.merge(all_alpha_df, file_mapping, on='Filename', how='left')
        
        # 精準對齊你的資料庫欄位 (不傳 feature_id 因為它是主鍵會自動遞增)
        db_columns = ['patient_id', 'file_id', 'average_power', 'PAF', 'CAF']
        upload_alpha_data = final_alpha_table[db_columns]
        
        success_count = 0
        for _, row in upload_alpha_data.iterrows():
            cols = ", ".join([f"`{c}`" for c in row.index])
            placeholders = ", ".join(["%s"] * len(row))
            sql = f"INSERT INTO Alpha_band ({cols}) VALUES ({placeholders})"
            
            # 資料型態防呆轉換
            val_list = []
            for c, v in row.items():
                if pd.isna(v) or v is None:
                    val_list.append(None)
                elif c == 'file_id':
                    val_list.append(int(v))
                elif c == 'patient_id':
                    val_list.append(str(v))
                else:
                    val_list.append(float(v))
            
            cursor.execute(sql, tuple(val_list))
            success_count += 1
            
        print(f"🎉 Alpha_band 資料表成功導入 {success_count} 筆患者特徵數據！")

except Exception as e:
    print(f"❌ 匯入失敗，錯誤訊息: {e}")
finally:
    conn.close()

--- 已經成功將你前面算好的數據，整理為個別患者的 Alpha 欄位 ---
    Filename patient_id  average_power       PAF       CAF
12340089.edf   12340089       1.738782  9.277778  9.664832
B1949126.edf   B1949126       3.564434 10.055556 10.077640
B2202077.edf   B2202077       1.806446 11.111111 10.648575
B2257947.edf   B2257947      -4.295608 11.888889 10.359983
B2666604.edf   B2666604       3.420571 11.027778 10.150755
B2875111.edf   B2875111       5.062934  9.055556  9.546560
B3167473.edf   B3167473      -0.676276  8.472222  9.610013
B3882577.edf   B3882577      -4.507880 10.138889  9.995073
B6881436.edf   B6881436       0.930839 11.444444 10.473433
B7063090.edf   B7063090       0.235138 10.444444 10.042612
-----------------------------------------------------------

正在將 Alpha 特徵匯入 MySQL 資料庫...
🎉 Alpha_band 資料表成功導入 168 筆患者特徵數據！
